# Glucose & HbA1c TFT Training with Explainability

This notebook trains a Temporal Fusion Transformer to forecast glucose (g_mean_use) and HbA1c from our synthetic dataset with integrated exercise minutes, regimen-based adherence, and clinical factors.

**Key Features:**
- Multi-target forecasting: glucose + HbA1c
- Attention-based explainability  
- Clinical factor importance
- Quantile regression for uncertainty


In [ ]:
# --- Environment Setup and Dependencies ---
import os
from pathlib import Path

# Create necessary directories
directories = [
    'models',
    'test_payloads', 
    'results',
    'scripts'
]

for directory in directories:
    Path(directory).mkdir(parents=True, exist_ok=True)
    print(f"✅ Directory created/verified: {directory}")

# Install required dependencies
%pip install numpy pandas matplotlib seaborn torch torchvision scikit-learn

print("🚀 Environment setup complete! Ready for Colab execution.")


In [ ]:
# --- Data Loading and Configuration ---
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

# Configuration for flexible input/output lengths
CONFIG = {
    'min_encoder_length': 45,    # Minimum input days
    'max_encoder_length': 180,   # Maximum input days  
    'prediction_length': 90,     # Fixed 90-day forecast
    'batch_size': 32,
    'learning_rate': 0.001,
    'epochs': 50,
    'quantiles': [0.1, 0.5, 0.9],  # 10th, 50th, 90th percentiles
    'hidden_size': 64,
    'n_heads': 4,
    'n_layers': 2,
    'dropout': 0.1
}

print("Configuration loaded:")
for k, v in CONFIG.items():
    print(f"  {k}: {v}")

# Load the augmented dataset
# Check multiple possible locations for flexibility
possible_paths = [
    "test_payloads/sim_glucose_hba1c_augmented.csv",  # Colab/current directory
    "../test_payloads/sim_glucose_hba1c_augmented.csv",  # Local relative path
    "/content/test_payloads/sim_glucose_hba1c_augmented.csv"  # Colab absolute path
]

data_path = None
for path in possible_paths:
    if Path(path).exists():
        data_path = Path(path)
        break

if data_path is None:
    print("❌ Dataset not found! Please upload 'sim_glucose_hba1c_augmented.csv' to the 'test_payloads' directory.")
    print("Expected locations:")
    for path in possible_paths:
        print(f"  - {path}")
    raise FileNotFoundError("Dataset file not found")

print(f"\n✅ Loading data from: {data_path}")
df = pd.read_csv(data_path)

print(f"Data shape: {df.shape}")
print(f"Date range: {df['date'].min()} to {df['date'].max()}")
print(f"Patients: {df['patient_id'].nunique()}")
print(f"Days per patient: {df.groupby('patient_id').size().describe()}")

# Convert date column
df['date'] = pd.to_datetime(df['date'])
df = df.sort_values(['patient_id', 'date']).reset_index(drop=True)

print("✅ Data loaded successfully!")


In [ ]:
# --- Feature Engineering and Categorization ---

# Define feature categories for TFT architecture
STATIC_FEATURES = [
    'patient_id', 'age', 'sex', 'height_cm', 'weight_kg', 'bmi',
    'has_CVD', 'has_CKD', 'has_chronic_allergy', 'alcohol_user', 'smoker'
]

TIME_VARYING_KNOWN = [
    'weekday', 'is_weekend'  # Known future values
]

TIME_VARYING_UNKNOWN = [
    # Core glucose/insulin features
    'g_mean_use', 'HbA1c', 'insulin_dose', 'insulin_adherence', 
    'insulin_dose_lag1', 'insulin_dose_lag2', 'insulin_dose_lag3',
    
    # Lifestyle and health factors
    'exercise_minutes', 'sleep_quality', 'meal_variability', 'stress_index',
    
    # Episode flags
    'illness_flag', 'alcohol_day', 'smoking_day', 'allergy_flare_day', 
    'alcohol_rebound_day', 'stress_high_day',
    
    # Glucose statistics
    'g_std', 'pct_hypo_adj', 'pct_hyper_adj', 'hypo_past7d', 'hyper_past7d',
    
    # Rolling features
    'g_mean_lag1', 'g_mean_7d_mean', 'g_mean_14d_std', 'g_mean_14d_mean', 'g_mean_30d_mean'
]

# Target variables (we'll forecast both glucose and HbA1c)
TARGETS = ['g_mean_use', 'HbA1c']

print("Feature categorization:")
print(f"Static features ({len(STATIC_FEATURES)}): {STATIC_FEATURES[:5]}...")
print(f"Time-varying known ({len(TIME_VARYING_KNOWN)}): {TIME_VARYING_KNOWN}")
print(f"Time-varying unknown ({len(TIME_VARYING_UNKNOWN)}): {TIME_VARYING_UNKNOWN[:5]}...")
print(f"Targets ({len(TARGETS)}): {TARGETS}")

# Verify all features exist in data
missing_features = []
all_features = STATIC_FEATURES + TIME_VARYING_KNOWN + TIME_VARYING_UNKNOWN
for feature in all_features:
    if feature not in df.columns:
        missing_features.append(feature)

if missing_features:
    print(f"\n⚠️  Missing features: {missing_features}")
else:
    print("\n✅ All features found in dataset!")


In [ ]:
# --- Sequence Creation for Variable Length Inputs ---

def create_sequences_variable_length(df, patient_ids, min_length=45, max_length=180, pred_length=90):
    """
    Create sequences with variable encoder length (45-180 days) and fixed prediction length (90 days)
    
    Args:
        df: DataFrame with patient data
        patient_ids: List of patient IDs to process
        min_length: Minimum encoder sequence length
        max_length: Maximum encoder sequence length  
        pred_length: Prediction sequence length (fixed)
    
    Returns:
        sequences: List of sequence dictionaries
    """
    sequences = []
    
    for patient_id in patient_ids:
        patient_data = df[df['patient_id'] == patient_id].sort_values('date').reset_index(drop=True)
        
        if len(patient_data) < min_length + pred_length:
            continue  # Skip patients with insufficient data
            
        # Generate multiple sequences per patient with different encoder lengths
        max_possible_length = min(max_length, len(patient_data) - pred_length)
        
        # Create sequences with different encoder lengths for robustness
        encoder_lengths = [min_length, min_length + 30, min_length + 60, max_possible_length]
        encoder_lengths = [l for l in encoder_lengths if l <= max_possible_length and l >= min_length]
        encoder_lengths = list(set(encoder_lengths))  # Remove duplicates
        
        for enc_len in encoder_lengths:
            # Multiple starting points for each encoder length
            max_start = len(patient_data) - enc_len - pred_length
            if max_start < 0:
                continue
                
            # Sample starting points (don't use all to avoid overfitting)
            n_samples = min(3, max_start + 1)  # Max 3 sequences per encoder length
            start_indices = np.linspace(0, max_start, n_samples, dtype=int)
            
            for start_idx in start_indices:
                end_encoder = start_idx + enc_len
                end_pred = end_encoder + pred_length
                
                if end_pred > len(patient_data):
                    continue
                
                # Extract sequences
                encoder_data = patient_data.iloc[start_idx:end_encoder]
                decoder_data = patient_data.iloc[end_encoder:end_pred]
                
                # Static features (patient-level, constant)
                static_features = encoder_data[STATIC_FEATURES].iloc[0].values
                
                # Time-varying features
                encoder_known = encoder_data[TIME_VARYING_KNOWN].values
                encoder_unknown = encoder_data[TIME_VARYING_UNKNOWN].values
                
                # Future known features (for decoder)
                decoder_known = decoder_data[TIME_VARYING_KNOWN].values
                
                # Targets (what we want to predict)
                targets = decoder_data[TARGETS].values
                
                sequence = {
                    'patient_id': patient_id,
                    'encoder_length': enc_len,
                    'static': static_features,
                    'encoder_known': encoder_known,
                    'encoder_unknown': encoder_unknown,
                    'decoder_known': decoder_known,
                    'targets': targets,
                    'start_date': encoder_data['date'].iloc[0],
                    'end_date': decoder_data['date'].iloc[-1]
                }
                
                sequences.append(sequence)
    
    return sequences

print("Creating sequences with variable encoder lengths...")

# Get patient list and split into train/val/test
all_patients = df['patient_id'].unique()
train_patients, temp_patients = train_test_split(all_patients, test_size=0.4, random_state=42)
val_patients, test_patients = train_test_split(temp_patients, test_size=0.5, random_state=42)

print(f"Patient split: {len(train_patients)} train, {len(val_patients)} val, {len(test_patients)} test")

# Create sequences
train_sequences = create_sequences_variable_length(
    df, train_patients, CONFIG['min_encoder_length'], CONFIG['max_encoder_length'], CONFIG['prediction_length']
)
val_sequences = create_sequences_variable_length(
    df, val_patients, CONFIG['min_encoder_length'], CONFIG['max_encoder_length'], CONFIG['prediction_length']
)
test_sequences = create_sequences_variable_length(
    df, test_patients, CONFIG['min_encoder_length'], CONFIG['max_encoder_length'], CONFIG['prediction_length']
)

print(f"Sequences created: {len(train_sequences)} train, {len(val_sequences)} val, {len(test_sequences)} test")
print(f"Example encoder lengths: {[seq['encoder_length'] for seq in train_sequences[:10]]}")

print("✅ Variable-length sequences created successfully!")


In [ ]:
# --- Data Scaling and Dataset Class ---

class TFTDataset(torch.utils.data.Dataset):
    """
    PyTorch Dataset for TFT with variable-length sequences and proper scaling
    """
    def __init__(self, sequences, scalers=None, fit_scalers=False):
        self.sequences = sequences
        self.scalers = scalers or {}
        
        if fit_scalers:
            self._fit_scalers()
        
        self._scale_sequences()
    
    def _fit_scalers(self):
        """Fit scalers on training data"""
        print("Fitting scalers...")
        
        # Collect all data for fitting scalers
        all_static = []
        all_encoder_known = []
        all_encoder_unknown = []
        all_decoder_known = []
        all_targets = []
        
        for seq in self.sequences:
            all_static.append(seq['static'])
            all_encoder_known.append(seq['encoder_known'])
            all_encoder_unknown.append(seq['encoder_unknown'])
            all_decoder_known.append(seq['decoder_known'])
            all_targets.append(seq['targets'])
        
        # Fit scalers
        self.scalers['static'] = StandardScaler()
        self.scalers['encoder_known'] = StandardScaler()
        self.scalers['encoder_unknown'] = StandardScaler()
        self.scalers['decoder_known'] = StandardScaler()
        self.scalers['targets'] = StandardScaler()
        
        # Static features (2D: n_samples x n_features)
        static_data = np.array(all_static)
        # Handle categorical features (convert to numeric first)
        static_numeric = np.zeros_like(static_data, dtype=float)
        for i, seq in enumerate(self.sequences):
            static_row = seq['static'].copy()
            # Convert patient_id to numeric (use index)
            if isinstance(static_row[0], str):
                static_row[0] = float(i)  # Simple patient encoding
            # Convert sex to numeric (M=1, F=0)
            if len(static_row) > 2 and isinstance(static_row[2], str):
                static_row[2] = 1.0 if static_row[2] == 'M' else 0.0
            static_numeric[i] = static_row.astype(float)
        
        self.scalers['static'].fit(static_numeric)
        
        # Time-varying features (3D: n_samples x n_timesteps x n_features -> 2D for fitting)
        encoder_known_data = np.concatenate(all_encoder_known, axis=0)
        encoder_unknown_data = np.concatenate(all_encoder_unknown, axis=0)
        decoder_known_data = np.concatenate(all_decoder_known, axis=0)
        targets_data = np.concatenate(all_targets, axis=0)
        
        self.scalers['encoder_known'].fit(encoder_known_data)
        self.scalers['encoder_unknown'].fit(encoder_unknown_data)
        self.scalers['decoder_known'].fit(decoder_known_data)
        self.scalers['targets'].fit(targets_data)
        
        print("✅ Scalers fitted!")
    
    def _scale_sequences(self):
        """Apply scaling to all sequences"""
        print("Scaling sequences...")
        
        for i, seq in enumerate(self.sequences):
            # Scale static features
            static_numeric = seq['static'].copy().astype(float)
            # Handle categorical encoding
            if len(STATIC_FEATURES) > 2:  # Has sex column
                static_numeric[2] = 1.0 if seq['static'][2] == 'M' else 0.0  # sex
            static_numeric[0] = float(i)  # patient_id as index
            
            seq['static_scaled'] = self.scalers['static'].transform(static_numeric.reshape(1, -1))[0]
            
            # Scale time-varying features
            seq['encoder_known_scaled'] = self.scalers['encoder_known'].transform(seq['encoder_known'])
            seq['encoder_unknown_scaled'] = self.scalers['encoder_unknown'].transform(seq['encoder_unknown'])
            seq['decoder_known_scaled'] = self.scalers['decoder_known'].transform(seq['decoder_known'])
            seq['targets_scaled'] = self.scalers['targets'].transform(seq['targets'])
        
        print("✅ Sequences scaled!")
    
    def __len__(self):
        return len(self.sequences)
    
    def __getitem__(self, idx):
        seq = self.sequences[idx]
        
        # Pad sequences to max length for batching
        max_enc_len = CONFIG['max_encoder_length']
        pred_len = CONFIG['prediction_length']
        
        # Encoder sequences (pad to max_enc_len)
        enc_len = seq['encoder_length']
        encoder_known = seq['encoder_known_scaled']
        encoder_unknown = seq['encoder_unknown_scaled']
        
        # Pad encoder sequences
        if enc_len < max_enc_len:
            pad_len = max_enc_len - enc_len
            encoder_known_padded = np.concatenate([
                np.zeros((pad_len, encoder_known.shape[1])), encoder_known
            ], axis=0)
            encoder_unknown_padded = np.concatenate([
                np.zeros((pad_len, encoder_unknown.shape[1])), encoder_unknown  
            ], axis=0)
        else:
            encoder_known_padded = encoder_known
            encoder_unknown_padded = encoder_unknown
        
        return {
            'encoder_known': torch.tensor(encoder_known_padded, dtype=torch.float32),
            'encoder_unknown': torch.tensor(encoder_unknown_padded, dtype=torch.float32),
            'decoder_known': torch.tensor(seq['decoder_known_scaled'], dtype=torch.float32),
            'static': torch.tensor(seq['static_scaled'], dtype=torch.float32),
            'targets': torch.tensor(seq['targets_scaled'], dtype=torch.float32),
            'encoder_length': torch.tensor(enc_len, dtype=torch.long),
            'patient_id': seq['patient_id']
        }

# Create datasets
print("Creating TFT datasets...")
train_dataset = TFTDataset(train_sequences, fit_scalers=True)
val_dataset = TFTDataset(val_sequences, scalers=train_dataset.scalers)
test_dataset = TFTDataset(test_sequences, scalers=train_dataset.scalers)

print(f"Dataset sizes: {len(train_dataset)} train, {len(val_dataset)} val, {len(test_dataset)} test")

# Create data loaders
train_loader = torch.utils.data.DataLoader(train_dataset, batch_size=CONFIG['batch_size'], shuffle=True)
val_loader = torch.utils.data.DataLoader(val_dataset, batch_size=CONFIG['batch_size'], shuffle=False)
test_loader = torch.utils.data.DataLoader(test_dataset, batch_size=CONFIG['batch_size'], shuffle=False)

print("✅ Datasets and loaders created!")


In [ ]:
# --- Data Validation and Testing ---

print("🔍 COMPREHENSIVE DATA VALIDATION")
print("=" * 50)

# 1. Test data loader
print("\n1. Testing data loaders...")
try:
    # Test train loader
    train_batch = next(iter(train_loader))
    print(f"✅ Train loader working: batch size = {train_batch['encoder_known'].shape[0]}")
    
    # Test val loader  
    val_batch = next(iter(val_loader))
    print(f"✅ Validation loader working: batch size = {val_batch['encoder_known'].shape[0]}")
    
    # Test test loader
    test_batch = next(iter(test_loader))
    print(f"✅ Test loader working: batch size = {test_batch['encoder_known'].shape[0]}")
    
except Exception as e:
    print(f"❌ Data loader error: {e}")
    raise

# 2. Validate batch shapes
print(f"\n2. Validating batch shapes...")
print(f"Encoder known: {train_batch['encoder_known'].shape} (should be: [batch, {CONFIG['max_encoder_length']}, {len(TIME_VARYING_KNOWN)}])")
print(f"Encoder unknown: {train_batch['encoder_unknown'].shape} (should be: [batch, {CONFIG['max_encoder_length']}, {len(TIME_VARYING_UNKNOWN)}])")
print(f"Decoder known: {train_batch['decoder_known'].shape} (should be: [batch, {CONFIG['prediction_length']}, {len(TIME_VARYING_KNOWN)}])")
print(f"Static: {train_batch['static'].shape} (should be: [batch, {len(STATIC_FEATURES)}])")
print(f"Targets: {train_batch['targets'].shape} (should be: [batch, {CONFIG['prediction_length']}, {len(TARGETS)}])")
print(f"Encoder lengths: {train_batch['encoder_length'].shape} (should be: [batch])")

# 3. Check for NaN values
print(f"\n3. Checking for NaN values...")
for key, tensor in train_batch.items():
    if key != 'patient_id':
        nan_count = torch.isnan(tensor).sum().item()
        if nan_count > 0:
            print(f"❌ Found {nan_count} NaN values in {key}")
        else:
            print(f"✅ No NaN values in {key}")

# 4. Check data ranges
print(f"\n4. Checking data ranges...")
print(f"Encoder lengths range: {train_batch['encoder_length'].min().item():.0f} - {train_batch['encoder_length'].max().item():.0f}")
print(f"Static features range: {train_batch['static'].min().item():.3f} - {train_batch['static'].max().item():.3f}")
print(f"Targets range: {train_batch['targets'].min().item():.3f} - {train_batch['targets'].max().item():.3f}")

# 5. Test model forward pass
print(f"\n5. Testing model forward pass...")
try:
    model_test = model.to('cpu')  # Use CPU for testing
    with torch.no_grad():
        predictions, attention_weights = model_test(
            train_batch['encoder_known'][:2],  # Small batch for testing
            train_batch['encoder_unknown'][:2],
            train_batch['decoder_known'][:2], 
            train_batch['static'][:2],
            train_batch['encoder_length'][:2]
        )
    
    print(f"✅ Model forward pass successful!")
    print(f"   Predictions shape: {predictions.shape} (should be: [2, {CONFIG['prediction_length']}, {len(TARGETS)}, {len(CONFIG['quantiles'])}])")
    print(f"   Attention layers: {len(attention_weights)}")
    print(f"   Attention shape: {attention_weights[0].shape} (should be: [2, {CONFIG['n_heads']}, seq_len, seq_len])")
    
except Exception as e:
    print(f"❌ Model forward pass failed: {e}")
    raise

# 6. Test loss calculation
print(f"\n6. Testing loss calculation...")
try:
    criterion_test = QuantileLoss(CONFIG['quantiles'])
    loss = criterion_test(predictions, train_batch['targets'][:2])
    print(f"✅ Loss calculation successful: {loss.item():.6f}")
except Exception as e:
    print(f"❌ Loss calculation failed: {e}")
    raise

# 7. Dataset statistics
print(f"\n7. Dataset statistics...")
print(f"Training sequences: {len(train_sequences)}")
print(f"Validation sequences: {len(val_sequences)}")
print(f"Test sequences: {len(test_sequences)}")

# Show encoder length distribution
encoder_lengths = [seq['encoder_length'] for seq in train_sequences]
print(f"Encoder length distribution:")
print(f"  Min: {min(encoder_lengths)} days")
print(f"  Max: {max(encoder_lengths)} days") 
print(f"  Mean: {sum(encoder_lengths)/len(encoder_lengths):.1f} days")

print(f"\n🎉 ALL DATA VALIDATION CHECKS PASSED!")
print(f"Ready to start training! 🚀")


In [ ]:
# --- TFT Model Architecture (Adapted from Successful Pattern) ---

class TemporalFusionTransformer(nn.Module):
    """
    TFT model adapted for glucose/HbA1c forecasting with variable encoder lengths
    Based on the successful architecture pattern from the existing notebook
    """
    def __init__(self, n_encoder_known, n_encoder_unknown, n_decoder_known, n_static, 
                 n_targets=2, hidden_size=64, n_heads=4, n_layers=2, dropout=0.1, n_quantiles=3):
        super().__init__()
        
        self.hidden_size = hidden_size
        self.n_heads = n_heads
        self.n_layers = n_layers
        self.n_quantiles = n_quantiles
        self.n_targets = n_targets
        
        # Input projections
        self.encoder_known_proj = nn.Linear(n_encoder_known, hidden_size)
        self.encoder_unknown_proj = nn.Linear(n_encoder_unknown, hidden_size)
        self.decoder_known_proj = nn.Linear(n_decoder_known, hidden_size)
        self.static_proj = nn.Linear(n_static, hidden_size)
        
        # Static context vector
        self.static_context = nn.Sequential(
            nn.Linear(hidden_size, hidden_size),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_size, hidden_size)
        )
        
        # LSTM encoder for variable-length sequences
        self.encoder_lstm = nn.LSTM(
            hidden_size * 2,  # encoder_known + encoder_unknown
            hidden_size,
            num_layers=2,
            batch_first=True,
            dropout=dropout if n_layers > 1 else 0,
            bidirectional=False
        )
        
        # Multi-head attention layers
        self.attention_layers = nn.ModuleList([
            nn.MultiheadAttention(hidden_size, n_heads, dropout=dropout, batch_first=True)
            for _ in range(n_layers)
        ])
        
        # Layer normalization
        self.layer_norms = nn.ModuleList([
            nn.LayerNorm(hidden_size) for _ in range(n_layers)
        ])
        
        # Decoder LSTM
        self.decoder_lstm = nn.LSTM(
            hidden_size,  # decoder_known + context
            hidden_size,
            num_layers=2,
            batch_first=True,
            dropout=dropout if n_layers > 1 else 0
        )
        
        # Output projection for quantile regression
        self.output_proj = nn.Linear(hidden_size, n_targets * n_quantiles)
        
        # Dropout
        self.dropout = nn.Dropout(dropout)
        
        # Initialize weights
        self._init_weights()
    
    def _init_weights(self):
        """Initialize weights using Xavier/Glorot initialization"""
        for module in self.modules():
            if isinstance(module, nn.Linear):
                nn.init.xavier_uniform_(module.weight)
                if module.bias is not None:
                    nn.init.zeros_(module.bias)
            elif isinstance(module, nn.LSTM):
                for name, param in module.named_parameters():
                    if 'weight' in name:
                        nn.init.xavier_uniform_(param)
                    elif 'bias' in name:
                        nn.init.zeros_(param)
    
    def forward(self, encoder_known, encoder_unknown, decoder_known, static, encoder_lengths=None):
        """
        Forward pass
        
        Args:
            encoder_known: (batch_size, max_encoder_len, n_encoder_known)
            encoder_unknown: (batch_size, max_encoder_len, n_encoder_unknown)  
            decoder_known: (batch_size, pred_len, n_decoder_known)
            static: (batch_size, n_static)
            encoder_lengths: (batch_size,) actual lengths for variable sequences
        
        Returns:
            predictions: (batch_size, pred_len, n_targets * n_quantiles)
        """
        batch_size = encoder_known.size(0)
        max_encoder_len = encoder_known.size(1)
        pred_len = decoder_known.size(1)
        
        # Project inputs
        enc_known = self.encoder_known_proj(encoder_known)  # (B, T_enc, H)
        enc_unknown = self.encoder_unknown_proj(encoder_unknown)  # (B, T_enc, H)
        dec_known = self.decoder_known_proj(decoder_known)  # (B, T_pred, H)
        static_emb = self.static_proj(static)  # (B, H)
        
        # Static context
        context = self.static_context(static_emb)  # (B, H)
        
        # Combine encoder inputs
        encoder_input = torch.cat([enc_known, enc_unknown], dim=-1)  # (B, T_enc, 2H)
        
        # LSTM encoding with variable lengths
        if encoder_lengths is not None:
            # Pack padded sequences for efficiency
            packed_input = nn.utils.rnn.pack_padded_sequence(
                encoder_input, encoder_lengths.cpu(), batch_first=True, enforce_sorted=False
            )
            packed_output, (hidden, cell) = self.encoder_lstm(packed_input)
            encoder_output, _ = nn.utils.rnn.pad_packed_sequence(packed_output, batch_first=True)
        else:
            encoder_output, (hidden, cell) = self.encoder_lstm(encoder_input)
        
        encoder_output = self.dropout(encoder_output)  # (B, T_enc, H)
        
        # Multi-head attention layers
        attn_output = encoder_output
        attention_weights = []
        
        for i, (attn_layer, norm_layer) in enumerate(zip(self.attention_layers, self.layer_norms)):
            # Self-attention with residual connection
            attn_out, attn_weights = attn_layer(attn_output, attn_output, attn_output)
            attn_output = norm_layer(attn_output + self.dropout(attn_out))
            attention_weights.append(attn_weights)
        
        # Global context from attention output (mean pooling over valid sequence length)
        if encoder_lengths is not None:
            # Mask out padded positions
            mask = torch.arange(max_encoder_len)[None, :].to(encoder_lengths.device) < encoder_lengths[:, None]
            mask = mask.unsqueeze(-1).float()  # (B, T_enc, 1)
            
            masked_output = attn_output * mask
            global_context = masked_output.sum(dim=1) / mask.sum(dim=1)  # (B, H)
        else:
            global_context = attn_output.mean(dim=1)  # (B, H)
        
        # Combine with static context
        combined_context = context + global_context  # (B, H)
        
        # Decoder with known future inputs
        decoder_input = dec_known + combined_context.unsqueeze(1)  # (B, T_pred, H)
        
        # Initialize decoder hidden state with encoder final state
        decoder_output, _ = self.decoder_lstm(decoder_input, (hidden, cell))
        decoder_output = self.dropout(decoder_output)  # (B, T_pred, H)
        
        # Output projection for quantile predictions
        predictions = self.output_proj(decoder_output)  # (B, T_pred, n_targets * n_quantiles)
        
        # Reshape for quantile outputs
        predictions = predictions.view(batch_size, pred_len, self.n_targets, self.n_quantiles)
        
        return predictions, attention_weights

# Initialize model
n_encoder_known = len(TIME_VARYING_KNOWN)
n_encoder_unknown = len(TIME_VARYING_UNKNOWN) 
n_decoder_known = len(TIME_VARYING_KNOWN)
n_static = len(STATIC_FEATURES)

print(f"Model input dimensions:")
print(f"  Encoder known: {n_encoder_known}")
print(f"  Encoder unknown: {n_encoder_unknown}")
print(f"  Decoder known: {n_decoder_known}")
print(f"  Static: {n_static}")
print(f"  Targets: {len(TARGETS)}")

model = TemporalFusionTransformer(
    n_encoder_known=n_encoder_known,
    n_encoder_unknown=n_encoder_unknown,
    n_decoder_known=n_decoder_known,
    n_static=n_static,
    n_targets=len(TARGETS),
    hidden_size=CONFIG['hidden_size'],
    n_heads=CONFIG['n_heads'],
    n_layers=CONFIG['n_layers'],
    dropout=CONFIG['dropout'],
    n_quantiles=len(CONFIG['quantiles'])
)

# Count parameters
total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

print(f"\nModel created!")
print(f"Total parameters: {total_params:,}")
print(f"Trainable parameters: {trainable_params:,}")
print("✅ TFT model architecture ready!")


In [ ]:
# --- Quantile Loss and Training Setup ---

class QuantileLoss(nn.Module):
    """
    Quantile loss for probabilistic forecasting
    Adapted from the successful pattern in the existing notebook
    """
    def __init__(self, quantiles):
        super().__init__()
        self.quantiles = quantiles
    
    def forward(self, predictions, targets):
        """
        Args:
            predictions: (batch_size, seq_len, n_targets, n_quantiles)
            targets: (batch_size, seq_len, n_targets)
        
        Returns:
            loss: scalar tensor
        """
        batch_size, seq_len, n_targets, n_quantiles = predictions.shape
        
        # Expand targets to match quantile dimension
        targets_expanded = targets.unsqueeze(-1)  # (B, T, n_targets, 1)
        
        # Calculate quantile loss for each quantile
        losses = []
        for i, q in enumerate(self.quantiles):
            pred_q = predictions[:, :, :, i]  # (B, T, n_targets)
            target_q = targets  # (B, T, n_targets)
            
            error = target_q - pred_q
            loss_q = torch.maximum(q * error, (q - 1) * error)
            losses.append(loss_q.mean())
        
        return torch.stack(losses).mean()

# Initialize loss function and optimizer
device = torch.device('cuda' if torch.cuda.is_available() else 'mps' if torch.mps.is_available() else 'cpu')
print(f"Using device: {device}")

model = model.to(device)
criterion = QuantileLoss(CONFIG['quantiles'])
optimizer = torch.optim.Adam(model.parameters(), lr=CONFIG['learning_rate'])
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', patience=5, factor=0.5)

# Mixed precision training (following successful pattern)
scaler = torch.cuda.amp.GradScaler() if device.type == 'cuda' else None

print("✅ Training setup complete!")
print(f"Device: {device}")
print(f"Loss function: Quantile loss with {len(CONFIG['quantiles'])} quantiles")
print(f"Optimizer: Adam (lr={CONFIG['learning_rate']})")
print(f"Scheduler: ReduceLROnPlateau")
print(f"Mixed precision: {'Enabled' if scaler else 'Disabled'}")


In [ ]:
# --- Training Loop with Early Stopping ---

def train_epoch(model, train_loader, criterion, optimizer, device, scaler=None):
    """Train for one epoch"""
    model.train()
    total_loss = 0
    n_batches = 0
    
    for batch in train_loader:
        encoder_known = batch['encoder_known'].to(device)
        encoder_unknown = batch['encoder_unknown'].to(device)
        decoder_known = batch['decoder_known'].to(device)
        static = batch['static'].to(device)
        targets = batch['targets'].to(device)
        encoder_lengths = batch['encoder_length'].to(device)
        
        optimizer.zero_grad()
        
        if scaler:
            with torch.cuda.amp.autocast():
                predictions, _ = model(encoder_known, encoder_unknown, decoder_known, static, encoder_lengths)
                loss = criterion(predictions, targets)
            
            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()
        else:
            predictions, _ = model(encoder_known, encoder_unknown, decoder_known, static, encoder_lengths)
            loss = criterion(predictions, targets)
            loss.backward()
            optimizer.step()
        
        total_loss += loss.item()
        n_batches += 1
    
    return total_loss / n_batches

def validate_epoch(model, val_loader, criterion, device):
    """Validate for one epoch"""
    model.eval()
    total_loss = 0
    n_batches = 0
    
    with torch.no_grad():
        for batch in val_loader:
            encoder_known = batch['encoder_known'].to(device)
            encoder_unknown = batch['encoder_unknown'].to(device)
            decoder_known = batch['decoder_known'].to(device)
            static = batch['static'].to(device)
            targets = batch['targets'].to(device)
            encoder_lengths = batch['encoder_length'].to(device)
            
            predictions, _ = model(encoder_known, encoder_unknown, decoder_known, static, encoder_lengths)
            loss = criterion(predictions, targets)
            
            total_loss += loss.item()
            n_batches += 1
    
    return total_loss / n_batches

# Training loop with early stopping
print("Starting training...")
print(f"Training for {CONFIG['epochs']} epochs")

train_losses = []
val_losses = []
best_val_loss = float('inf')
patience_counter = 0
patience = 10

for epoch in range(CONFIG['epochs']):
    # Train
    train_loss = train_epoch(model, train_loader, criterion, optimizer, device, scaler)
    
    # Validate
    val_loss = validate_epoch(model, val_loader, criterion, device)
    
    # Update scheduler
    scheduler.step(val_loss)
    
    # Record losses
    train_losses.append(train_loss)
    val_losses.append(val_loss)
    
    # Early stopping check
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        patience_counter = 0
        
        # Save best model (flexible path for Colab)
        model_save_path = 'models/glucose_hba1c_tft_best.pth'  # Current directory for Colab
        torch.save({
            'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'train_losses': train_losses,
            'val_losses': val_losses,
            'scalers': train_dataset.scalers,
            'config': CONFIG,
            'epoch': epoch,
            'best_val_loss': best_val_loss
        }, model_save_path)
        
    else:
        patience_counter += 1
    
    # Print progress
    if epoch % 5 == 0 or epoch == CONFIG['epochs'] - 1:
        current_lr = optimizer.param_groups[0]['lr']
        print(f"Epoch {epoch+1:3d}/{CONFIG['epochs']} | "
              f"Train Loss: {train_loss:.6f} | "
              f"Val Loss: {val_loss:.6f} | "
              f"Best Val: {best_val_loss:.6f} | "
              f"LR: {current_lr:.2e} | "
              f"Patience: {patience_counter}/{patience}")
    
    # Early stopping
    if patience_counter >= patience:
        print(f"Early stopping at epoch {epoch+1}")
        break

print("✅ Training completed!")
print(f"Best validation loss: {best_val_loss:.6f}")
print(f"Model saved as: {model_save_path}")


In [ ]:
# --- Evaluation and Explainability ---

def evaluate_model(model, test_loader, scalers, device, horizons=[7, 14, 30, 60, 90]):
    """
    Comprehensive evaluation with explainability for variable horizons up to 90 days
    """
    model.eval()
    all_predictions = []
    all_targets = []
    all_attention_weights = []
    
    with torch.no_grad():
        for batch in test_loader:
            encoder_known = batch['encoder_known'].to(device)
            encoder_unknown = batch['encoder_unknown'].to(device)
            decoder_known = batch['decoder_known'].to(device)
            static = batch['static'].to(device)
            targets = batch['targets'].to(device)
            encoder_lengths = batch['encoder_length'].to(device)
            
            predictions, attention_weights = model(encoder_known, encoder_unknown, decoder_known, static, encoder_lengths)
            
            # Denormalize predictions and targets
            batch_size, pred_len, n_targets, n_quantiles = predictions.shape
            
            # Reshape for inverse transform
            pred_reshaped = predictions.cpu().numpy().reshape(-1, n_targets * n_quantiles)
            target_reshaped = targets.cpu().numpy().reshape(-1, n_targets)
            
            # Denormalize (assuming targets scaler works for both glucose and HbA1c)
            pred_denorm = np.zeros_like(pred_reshaped)
            for i in range(n_targets):
                for j in range(n_quantiles):
                    col_idx = i * n_quantiles + j
                    pred_denorm[:, col_idx] = scalers['targets'].inverse_transform(
                        pred_reshaped[:, [col_idx]]
                    ).flatten()
            
            target_denorm = scalers['targets'].inverse_transform(target_reshaped)
            
            # Reshape back
            pred_denorm = pred_denorm.reshape(batch_size, pred_len, n_targets, n_quantiles)
            target_denorm = target_denorm.reshape(batch_size, pred_len, n_targets)
            
            all_predictions.append(pred_denorm)
            all_targets.append(target_denorm)
            all_attention_weights.append([aw.cpu().numpy() for aw in attention_weights])
    
    predictions = np.concatenate(all_predictions, axis=0)
    targets = np.concatenate(all_targets, axis=0)
    
    # Calculate metrics for each horizon and target
    results = {}
    target_names = ['Glucose', 'HbA1c']
    
    for h_idx, h in enumerate(horizons):
        if h <= predictions.shape[1]:  # Check if horizon is within prediction length
            results[h] = {}
            
            for t_idx, target_name in enumerate(target_names):
                # Get predictions and targets for this horizon and target
                pred_h_t = predictions[:, h-1, t_idx, :]  # (batch, quantiles)
                target_h_t = targets[:, h-1, t_idx]       # (batch,)
                
                # Basic regression metrics (using median prediction)
                mae = np.mean(np.abs(pred_h_t[:, 1] - target_h_t))
                rmse = np.sqrt(np.mean((pred_h_t[:, 1] - target_h_t) ** 2))
                mape = np.mean(np.abs((pred_h_t[:, 1] - target_h_t) / (target_h_t + 1e-8))) * 100
                
                # Coverage (90% confidence interval)
                lower = pred_h_t[:, 0]  # 10th percentile
                upper = pred_h_t[:, 2]  # 90th percentile
                coverage = np.mean((target_h_t >= lower) & (target_h_t <= upper))
                
                # Clinical metrics for glucose
                if target_name == 'Glucose':
                    # Hypoglycemia events (< 70 mg/dL)
                    hypo_actual = (target_h_t < 70).astype(int)
                    hypo_pred = (pred_h_t[:, 1] < 70).astype(int)
                    
                    # Hyperglycemia events (> 180 mg/dL)
                    hyper_actual = (target_h_t > 180).astype(int)
                    hyper_pred = (pred_h_t[:, 1] > 180).astype(int)
                    
                    # Time in range (70-180 mg/dL)
                    tir_actual = np.mean((target_h_t >= 70) & (target_h_t <= 180))
                    tir_pred = np.mean((pred_h_t[:, 1] >= 70) & (pred_h_t[:, 1] <= 180))
                    
                    clinical_metrics = {
                        'hypo_sensitivity': np.mean(hypo_pred[hypo_actual == 1]) if np.sum(hypo_actual) > 0 else 0,
                        'hyper_sensitivity': np.mean(hyper_pred[hyper_actual == 1]) if np.sum(hyper_actual) > 0 else 0,
                        'tir_actual': tir_actual,
                        'tir_predicted': tir_pred,
                        'tir_difference': abs(tir_actual - tir_pred)
                    }
                else:
                    clinical_metrics = {}
                
                results[h][target_name] = {
                    'mae': mae,
                    'rmse': rmse,
                    'mape': mape,
                    'coverage': coverage,
                    'predictions': pred_h_t,
                    'targets': target_h_t,
                    **clinical_metrics
                }
    
    return results, all_attention_weights

def plot_evaluation_results(results, train_losses, val_losses):
    """Plot comprehensive evaluation results"""
    horizons = list(results.keys())
    target_names = ['Glucose', 'HbA1c']
    
    fig, axes = plt.subplots(3, 3, figsize=(18, 15))
    
    # Plot 1: Training curves
    axes[0, 0].plot(train_losses, label='Training Loss', color='blue', alpha=0.7)
    axes[0, 0].plot(val_losses, label='Validation Loss', color='red', alpha=0.7)
    axes[0, 0].set_title('Training Progress')
    axes[0, 0].set_xlabel('Epoch')
    axes[0, 0].set_ylabel('Loss')
    axes[0, 0].legend()
    axes[0, 0].grid(True, alpha=0.3)
    
    # Plot 2: MAE by horizon for both targets
    for t_idx, target_name in enumerate(target_names):
        maes = [results[h][target_name]['mae'] for h in horizons]
        axes[0, 1].plot(horizons, maes, 'o-', label=f'{target_name} MAE', alpha=0.7)
    axes[0, 1].set_title('MAE by Forecast Horizon')
    axes[0, 1].set_xlabel('Horizon (days)')
    axes[0, 1].set_ylabel('MAE')
    axes[0, 1].legend()
    axes[0, 1].grid(True, alpha=0.3)
    
    # Plot 3: Coverage by horizon
    for t_idx, target_name in enumerate(target_names):
        coverages = [results[h][target_name]['coverage'] for h in horizons]
        axes[0, 2].plot(horizons, coverages, 's-', label=f'{target_name} Coverage', alpha=0.7)
    axes[0, 2].axhline(y=0.9, color='black', linestyle='--', alpha=0.5, label='Target 90%')
    axes[0, 2].set_title('Coverage by Forecast Horizon')
    axes[0, 2].set_xlabel('Horizon (days)')
    axes[0, 2].set_ylabel('Coverage')
    axes[0, 2].legend()
    axes[0, 2].grid(True, alpha=0.3)
    
    # Plot 4: Sample forecasts for 30-day horizon
    h = 30 if 30 in results else horizons[len(horizons)//2]
    for t_idx, target_name in enumerate(target_names):
        pred_h = results[h][target_name]['predictions']
        target_h = results[h][target_name]['targets']
        
        n_samples = min(50, len(pred_h))
        x = range(n_samples)
        
        ax = axes[1, t_idx]
        ax.plot(x, target_h[:n_samples], 'o-', label='Actual', color='black', alpha=0.7, markersize=3)
        ax.plot(x, pred_h[:n_samples, 1], 's-', label='Median Forecast', color='blue', markersize=3)
        ax.fill_between(x, pred_h[:n_samples, 0], pred_h[:n_samples, 2], 
                       alpha=0.3, color='blue', label='90% CI')
        ax.set_title(f'{target_name} - {h}-Day Forecast')
        ax.set_xlabel('Sample Index')
        ax.set_ylabel(f'{target_name} {"(mg/dL)" if target_name == "Glucose" else "(%)"}')
        ax.legend()
        ax.grid(True, alpha=0.3)
    
    # Plot 5: Clinical metrics for glucose (Time in Range)
    if 'tir_actual' in results[horizons[0]]['Glucose']:
        tir_actual = [results[h]['Glucose']['tir_actual'] for h in horizons]
        tir_pred = [results[h]['Glucose']['tir_predicted'] for h in horizons]
        
        axes[1, 2].plot(horizons, tir_actual, 'o-', label='Actual TIR', color='green', alpha=0.7)
        axes[1, 2].plot(horizons, tir_pred, 's-', label='Predicted TIR', color='orange', alpha=0.7)
        axes[1, 2].set_title('Time in Range (70-180 mg/dL)')
        axes[1, 2].set_xlabel('Horizon (days)')
        axes[1, 2].set_ylabel('TIR')
        axes[1, 2].legend()
        axes[1, 2].grid(True, alpha=0.3)
    
    # Plot 6-8: Error distributions for key horizons
    key_horizons = [7, 30, 90] if all(h in results for h in [7, 30, 90]) else horizons[:3]
    for i, h in enumerate(key_horizons):
        if i < 3:
            ax = axes[2, i]
            for t_idx, target_name in enumerate(target_names):
                pred_h = results[h][target_name]['predictions'][:, 1]  # Median predictions
                target_h = results[h][target_name]['targets']
                errors = pred_h - target_h
                
                ax.hist(errors, bins=30, alpha=0.6, label=f'{target_name} Error', density=True)
            
            ax.set_title(f'Prediction Errors - {h} Days')
            ax.set_xlabel('Prediction Error')
            ax.set_ylabel('Density')
            ax.legend()
            ax.grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()
    
    # Print summary statistics
    print("\\n" + "="*80)
    print("TFT MODEL EVALUATION RESULTS - 90-DAY FORECASTING")
    print("="*80)
    
    for h in horizons:
        print(f"\\nHorizon {h} days:")
        print("-" * 40)
        
        for target_name in target_names:
            r = results[h][target_name]
            print(f"{target_name}:")
            print(f"  MAE: {r['mae']:.3f}")
            print(f"  RMSE: {r['rmse']:.3f}")
            print(f"  MAPE: {r['mape']:.2f}%")
            print(f"  Coverage: {r['coverage']:.3f}")
            
            if 'tir_actual' in r:
                print(f"  Time in Range (Actual): {r['tir_actual']:.3f}")
                print(f"  Time in Range (Predicted): {r['tir_predicted']:.3f}")
                print(f"  TIR Difference: {r['tir_difference']:.3f}")

# Load best model and evaluate
print("Loading best model for evaluation...")
model_load_path = 'models/glucose_hba1c_tft_best.pth'  # Flexible path for Colab
checkpoint = torch.load(model_load_path, map_location=device)
model.load_state_dict(checkpoint['model_state_dict'])

print("Evaluating model on test set...")
results, attention_weights = evaluate_model(model, test_loader, train_dataset.scalers, device)

# Plot results
plot_evaluation_results(results, train_losses, val_losses)

print("✅ Evaluation completed with explainability features!")


In [ ]:
# --- Explainability Analysis ---

def extract_feature_importance(model, test_loader, scalers, device, n_samples=100):
    """
    Extract feature importance and attention patterns for explainability
    """
    model.eval()
    all_attention_weights = []
    all_static_features = []
    all_encoder_features = []
    all_predictions = []
    all_targets = []
    
    sample_count = 0
    
    with torch.no_grad():
        for batch in test_loader:
            if sample_count >= n_samples:
                break
                
            encoder_known = batch['encoder_known'].to(device)
            encoder_unknown = batch['encoder_unknown'].to(device)
            decoder_known = batch['decoder_known'].to(device)
            static = batch['static'].to(device)
            targets = batch['targets'].to(device)
            encoder_lengths = batch['encoder_length'].to(device)
            
            # Forward pass with attention extraction
            predictions, attention_weights = model(encoder_known, encoder_unknown, decoder_known, static, encoder_lengths)
            
            # Store data for analysis
            batch_size = encoder_known.size(0)
            for i in range(min(batch_size, n_samples - sample_count)):
                all_attention_weights.append([aw[i].cpu().numpy() for aw in attention_weights])
                all_static_features.append(static[i].cpu().numpy())
                all_encoder_features.append(torch.cat([encoder_known[i], encoder_unknown[i]], dim=-1).cpu().numpy())
                all_predictions.append(predictions[i].cpu().numpy())
                all_targets.append(targets[i].cpu().numpy())
                
            sample_count += batch_size
    
    return {
        'attention_weights': all_attention_weights,
        'static_features': all_static_features,
        'encoder_features': all_encoder_features,
        'predictions': all_predictions,
        'targets': all_targets
    }

def plot_attention_analysis(explainability_data, feature_names):
    """
    Plot attention patterns and feature importance
    """
    fig, axes = plt.subplots(2, 3, figsize=(18, 12))
    
    # 1. Average attention weights across all samples
    all_attention = explainability_data['attention_weights']
    n_layers = len(all_attention[0])
    
    # Average attention across samples and heads for first layer
    avg_attention = np.mean([attn[0] for attn in all_attention], axis=0)  # First layer
    avg_attention = np.mean(avg_attention, axis=0)  # Average across heads
    
    im1 = axes[0, 0].imshow(avg_attention, cmap='Blues', aspect='auto')
    axes[0, 0].set_title('Average Attention Weights (Layer 1)')
    axes[0, 0].set_xlabel('Key Position')
    axes[0, 0].set_ylabel('Query Position')
    plt.colorbar(im1, ax=axes[0, 0])
    
    # 2. Attention distribution by position
    attention_by_pos = np.mean(avg_attention, axis=0)  # Average attention received by each position
    axes[0, 1].plot(attention_by_pos, 'b-', linewidth=2)
    axes[0, 1].set_title('Attention by Sequence Position')
    axes[0, 1].set_xlabel('Sequence Position (Days)')
    axes[0, 1].set_ylabel('Average Attention Weight')
    axes[0, 1].grid(True, alpha=0.3)
    
    # 3. Feature importance from static features
    static_features = np.array(explainability_data['static_features'])
    static_importance = np.std(static_features, axis=0)  # Use std as importance measure
    
    axes[0, 2].bar(range(len(STATIC_FEATURES)), static_importance)
    axes[0, 2].set_title('Static Feature Importance (Variability)')
    axes[0, 2].set_xlabel('Feature Index')
    axes[0, 2].set_ylabel('Standard Deviation')
    axes[0, 2].set_xticks(range(len(STATIC_FEATURES)))
    axes[0, 2].set_xticklabels(STATIC_FEATURES, rotation=45, ha='right')
    
    # 4. Temporal attention patterns
    # Show how attention changes over prediction horizons
    attention_over_time = []
    for i in range(min(90, avg_attention.shape[0])):  # Up to 90 days
        attention_over_time.append(np.mean(avg_attention[i, :]))
    
    axes[1, 0].plot(attention_over_time, 'g-', linewidth=2)
    axes[1, 0].set_title('Attention Pattern Over Time')
    axes[1, 0].set_xlabel('Days into Sequence')
    axes[1, 0].set_ylabel('Average Attention')
    axes[1, 0].grid(True, alpha=0.3)
    
    # 5. Feature correlation with predictions
    encoder_features = np.array(explainability_data['encoder_features'])
    predictions = np.array(explainability_data['predictions'])
    
    # Calculate correlation between features and glucose predictions
    glucose_preds = predictions[:, -1, 0, 1]  # Last day, glucose, median prediction
    feature_correlations = []
    
    for feat_idx in range(encoder_features.shape[-1]):
        feat_values = encoder_features[:, -1, feat_idx]  # Last day features
        corr = np.corrcoef(feat_values, glucose_preds)[0, 1]
        feature_correlations.append(abs(corr) if not np.isnan(corr) else 0)
    
    all_feature_names = TIME_VARYING_KNOWN + TIME_VARYING_UNKNOWN
    top_features = sorted(zip(feature_correlations, all_feature_names), reverse=True)[:10]
    
    top_corrs, top_names = zip(*top_features)
    axes[1, 1].barh(range(len(top_corrs)), top_corrs)
    axes[1, 1].set_title('Top 10 Features by Correlation with Glucose')
    axes[1, 1].set_xlabel('Absolute Correlation')
    axes[1, 1].set_yticks(range(len(top_names)))
    axes[1, 1].set_yticklabels(top_names)
    
    # 6. Prediction uncertainty analysis
    pred_uncertainty = predictions[:, :, 0, 2] - predictions[:, :, 0, 0]  # Glucose uncertainty (90th - 10th percentile)
    uncertainty_over_horizon = np.mean(pred_uncertainty, axis=0)
    
    axes[1, 2].plot(uncertainty_over_horizon, 'r-', linewidth=2)
    axes[1, 2].set_title('Prediction Uncertainty Over Horizon')
    axes[1, 2].set_xlabel('Days Ahead')
    axes[1, 2].set_ylabel('Prediction Interval Width')
    axes[1, 2].grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()

def analyze_individual_prediction(explainability_data, sample_idx=0):
    """
    Analyze a single prediction for detailed explainability
    """
    attention = explainability_data['attention_weights'][sample_idx]
    static_feat = explainability_data['static_features'][sample_idx]
    encoder_feat = explainability_data['encoder_features'][sample_idx]
    prediction = explainability_data['predictions'][sample_idx]
    target = explainability_data['targets'][sample_idx]
    
    fig, axes = plt.subplots(2, 2, figsize=(15, 10))
    
    # 1. Individual attention heatmap
    layer_0_attention = attention[0]  # First attention layer
    avg_head_attention = np.mean(layer_0_attention, axis=0)  # Average across heads
    
    im = axes[0, 0].imshow(avg_head_attention, cmap='Blues', aspect='auto')
    axes[0, 0].set_title(f'Attention Heatmap - Sample {sample_idx}')
    axes[0, 0].set_xlabel('Key Position (Input Days)')
    axes[0, 0].set_ylabel('Query Position (Input Days)')
    plt.colorbar(im, ax=axes[0, 0])
    
    # 2. Key input features over time
    glucose_history = encoder_feat[:, TIME_VARYING_UNKNOWN.index('g_mean_use')]
    insulin_history = encoder_feat[:, TIME_VARYING_UNKNOWN.index('insulin_dose')]
    
    axes[0, 1].plot(glucose_history, 'b-', label='Glucose History', linewidth=2)
    axes[0, 1].set_ylabel('Glucose (scaled)', color='b')
    axes[0, 1].tick_params(axis='y', labelcolor='b')
    
    ax2 = axes[0, 1].twinx()
    ax2.plot(insulin_history, 'r-', label='Insulin Dose', linewidth=2)
    ax2.set_ylabel('Insulin (scaled)', color='r')
    ax2.tick_params(axis='y', labelcolor='r')
    
    axes[0, 1].set_title('Key Input Features Over Time')
    axes[0, 1].set_xlabel('Days')
    axes[0, 1].grid(True, alpha=0.3)
    
    # 3. Prediction vs actual with uncertainty
    days_ahead = range(1, len(prediction) + 1)
    glucose_pred_median = prediction[:, 0, 1]  # Glucose median
    glucose_pred_lower = prediction[:, 0, 0]   # Glucose 10th percentile
    glucose_pred_upper = prediction[:, 0, 2]   # Glucose 90th percentile
    glucose_actual = target[:, 0]              # Actual glucose
    
    axes[1, 0].plot(days_ahead, glucose_actual, 'ko-', label='Actual', linewidth=2, markersize=4)
    axes[1, 0].plot(days_ahead, glucose_pred_median, 'bo-', label='Predicted (Median)', linewidth=2, markersize=4)
    axes[1, 0].fill_between(days_ahead, glucose_pred_lower, glucose_pred_upper, 
                           alpha=0.3, color='blue', label='90% Confidence Interval')
    
    axes[1, 0].set_title('Glucose Prediction vs Actual')
    axes[1, 0].set_xlabel('Days Ahead')
    axes[1, 0].set_ylabel('Glucose (scaled)')
    axes[1, 0].legend()
    axes[1, 0].grid(True, alpha=0.3)
    
    # 4. Static feature values for this patient
    axes[1, 1].bar(range(len(STATIC_FEATURES)), static_feat)
    axes[1, 1].set_title('Patient Static Features')
    axes[1, 1].set_xlabel('Feature')
    axes[1, 1].set_ylabel('Value (scaled)')
    axes[1, 1].set_xticks(range(len(STATIC_FEATURES)))
    axes[1, 1].set_xticklabels(STATIC_FEATURES, rotation=45, ha='right')
    
    plt.tight_layout()
    plt.show()
    
    # Print interpretation
    print(f"\n🔍 INDIVIDUAL PREDICTION ANALYSIS - Sample {sample_idx}")
    print("=" * 60)
    print(f"Patient characteristics (scaled values):")
    for i, feat_name in enumerate(STATIC_FEATURES):
        print(f"  {feat_name}: {static_feat[i]:.3f}")
    
    # Calculate prediction accuracy
    glucose_mae = np.mean(np.abs(glucose_pred_median - glucose_actual))
    print(f"\nPrediction accuracy:")
    print(f"  Glucose MAE: {glucose_mae:.3f} (scaled units)")
    
    # Attention insights
    attention_focus = np.argmax(np.mean(avg_head_attention, axis=0))
    print(f"\nAttention insights:")
    print(f"  Model focuses most on day {attention_focus} of input sequence")
    print(f"  Recent days receive {np.mean(avg_head_attention[:, -10:]):.3f} average attention")
    print(f"  Early days receive {np.mean(avg_head_attention[:, :10]):.3f} average attention")

# Extract explainability data
print("🧠 EXTRACTING EXPLAINABILITY DATA...")
print("This may take a few minutes...")

explainability_data = extract_feature_importance(model, test_loader, train_dataset.scalers, device, n_samples=50)

print(f"✅ Explainability data extracted for {len(explainability_data['attention_weights'])} samples")

# Plot overall attention analysis
print("\n📊 PLOTTING ATTENTION ANALYSIS...")
plot_attention_analysis(explainability_data, TIME_VARYING_KNOWN + TIME_VARYING_UNKNOWN)

# Analyze individual predictions
print("\n🔍 INDIVIDUAL PREDICTION ANALYSIS...")
for sample_idx in [0, 1, 2]:  # Analyze first 3 samples
    analyze_individual_prediction(explainability_data, sample_idx)

print("\n🎯 EXPLAINABILITY SUMMARY:")
print("=" * 50)
print("✅ Attention weights show which input days the model focuses on")
print("✅ Feature importance reveals which patient characteristics matter most") 
print("✅ Individual analyses show model reasoning for specific predictions")
print("✅ Uncertainty quantification provides confidence intervals")
print("\n💡 Use these insights to understand and trust model predictions!")


# 🎯 **Colab Execution Instructions**

## **Before Running:**

1. **Upload Dataset**: Upload `sim_glucose_hba1c_augmented.csv` to the `test_payloads/` directory
   - The setup cell will create this directory automatically
   - You can drag and drop the file into the Colab file browser

2. **Runtime**: Use GPU runtime for faster training
   - Runtime → Change runtime type → GPU (T4 or better)

3. **Memory**: The model requires ~2-4GB RAM for training

## **Expected Training Time:**
- **With GPU**: ~15-30 minutes (50 epochs)
- **With CPU**: ~2-3 hours (not recommended)

## **Output Files:**
- `models/glucose_hba1c_tft_best.pth` - Best trained model
- Training progress plots and evaluation metrics

## **Key Features:**
- ✅ Variable input length: 45-180 days
- ✅ 90-day forecasting horizon  
- ✅ Multi-target: Glucose + HbA1c
- ✅ Quantile regression for uncertainty
- ✅ Clinical metrics (Time in Range, hypo/hyper events)
- ✅ Attention-based explainability

**Ready to train! 🚀**
